## What is **RAG** (Retrieval-Augmented Generation)?

RAG is an advanced AI technique that **combines two things**:

1. **Search (retrieval)**: Find relevant documents or text chunks.
2. **Generation**: Use a language model (like GPT) to generate a response using the retrieved info.

It’s like combining **Google + ChatGPT** in one system:

- Search like Google
- Answer like ChatGPT

**Self RAG**

**Self-RAG** (Self-Retrieval-Augmented Generation) means:
- The system retrieves documents and generates an answer.
- Then it ask the LLM to evaluate its own answer 
— checking if it's supported by the context or if it hallucinated.
- If the answer is **not good enough** → it retrieves again and regenerates.
- It's like a student who writes an answer, re-reads it, and says "wait, that's not backed by my notes" — then fixes it.

# How Self RAG Works?

1. User asks question  
2. Retrieve documents  
3. Generate answer  
4. LLM evaluates its own answer  
5. If bad → retrieve again / regenerate  
6. Final answer  

In [7]:
#Step 1: Imports
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
# Step 2: Load document.txt, split into chunks, and create embeddings

# Load the text file
loader = TextLoader("document.txt", encoding="utf-8")
documents = loader.load()

# Split into small chunks so the retriever can find relevant pieces
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
chunks = splitter.split_documents(documents)

# Convert chunks into embeddings and store in FAISS
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embedding=embeddings)
vectorstore.save_local("faiss_index")

# Create a retriever to search through the chunks
retriever = vectorstore.as_retriever()

In [9]:
#Step 3: LLM
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [10]:
# Self-RAG: The LLM generates an answer, then validates it, and improves if needed

def self_rag(query):

    # STEP 1: Retrieve relevant documents
    docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in docs])

    # STEP 2: Generate an initial answer
    answer = llm.invoke(f"""
    Answer the question using the context below.
    Context: {context}
    Question: {query}
    Answer:
    """).content.strip()

    print("Initial Answer:\n", answer)

    # STEP 3: ask the LLM to evaluate its own answer
    critique = llm.invoke(f"""
    Question: {query}
    Context: {context}
    Answer: {answer}

    Is the answer correct and fully supported by the context? Check for hallucination.
    Answer only YES or NO and explain briefly.
    """).content.strip()

    print("\nSelf-Critique:", critique)

    # STEP 4: If the answer is not good — retrieve again and regenerate
    if "NO" in critique.upper():
        print("\n--- Answer not good enough. Re-retrieving and regenerating... ---\n")
        docs = retriever.invoke(query + " detailed explanation")
        context = "\n".join([doc.page_content for doc in docs])

        answer = llm.invoke(f"""
        Answer the question using the context below. Be accurate and avoid hallucination.
        Context: {context}
        Question: {query}
        Improved Answer:
        """).content.strip()

        print("Improved Answer:\n", answer)

    return answer

# HOW IT WORKS (simple analogy):
# 1. You ask a question
# 2. The system searches its notes and writes an answer
# 3. It re-reads its answer and asks: "Is this correct and backed by my notes?" (YES/NO)
# 4. If NO → it searches again with better keywords and rewrites the answer
# 5. Returns the final (possibly improved) answer

In [11]:
query = "What is Self-RAG?"
result = self_rag(query)

print("\nFinal Answer:\n", result)

Initial Answer:
 Self-RAG is a type of Retrieval-Augmented Generation (RAG) where the language model (LLM) evaluates and improves its own output. This approach allows the model to enhance the quality and relevance of the information it generates by self-assessing its responses.

Self-Critique: YES. The answer correctly defines Self-RAG as a type of Retrieval-Augmented Generation (RAG) where the language model evaluates and improves its own output. This aligns with the context provided, which explicitly states that Self-RAG involves the LLM assessing its responses to enhance quality and relevance. There is no indication of hallucination in the answer.

--- Answer not good enough. Re-retrieving and regenerating... ---

Improved Answer:
 Self-RAG is a variant of the Retrieval-Augmented Generation (RAG) framework that focuses on enhancing the retrieval process by allowing the model to autonomously retrieve and utilize relevant information from external knowledge sources. This approach aims

## Corrective RAG (CRAG)

Corrective RAG (CRAG) checks whether the retrieved documents are good enough, and if not, it fixes the retrieval before answering.

**Problem:** Basic RAG blindly uses whatever documents it finds — even if they're **irrelevant**.

**Solution:** Before answering, **check if the retrieved docs are actually useful.**
- If YES → use them to answer
- If NO → **rewrite the query** and search again

# How CRAG Works?

1. User asks question  
2. System retrieves documents  
3. LLM evaluates: “Are these docs useful?”  
4. If NO → rewrite query / retrieve again  
5. Generate final answer  


| Aspect          | Corrective RAG   | Self-RAG           |
| --------------- | ---------------- | ------------------ |
| What is checked | 📄 Documents     | 🧠 Answer          |
| When it happens | Before answering | After answering    |
| Main goal       | Better context   | Better correctness |
| Type of control | Retrieval-level  | Generation-level   |


In [ ]:
# Corrective RAG (uses same tools as Self-RAG — no new libraries!)

def corrective_rag(query):

    # Step 1: Retrieve documents
    docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in docs])

    # Step 2: Ask LLM — are these docs relevant?
    check = llm.invoke(f"""
    Question: {query}
    Context: {context}
    Are these documents relevant to the question? Answer only YES or NO.
    """).content.strip()

    print("Are docs relevant?", check)

    # Step 3: If NOT relevant — rewrite the query and search again
    if "NO" in check:
        print("Rewriting query for better results...")
        better_query = llm.invoke(f"""
        The original query did not get good results. Rewrite it to be more specific.
        Original query: {query}
        Return only the rewritten query, nothing else.
        """).content.strip()

        print("New query:", better_query)
        docs = retriever.invoke(better_query)
        context = "\n".join([doc.page_content for doc in docs])

    # Step 4: Generate answer
    answer = llm.invoke(f"""
    Answer the question using only this context:
    Context: {context}
    Question: {query}
    """)
    return answer.content

In [ ]:
# Test: Query matching your documents
print(corrective_rag("What are the core components of RAG?"))

In [ ]:
# Test: Query NOT in your documents (triggers web search)
print(corrective_rag("What is the capital of France?"))

## Fusion RAG (RAG Fusion)
Fusion RAG improves search by asking the same question in multiple ways and combining the results.

## How it Works (Simple Steps)
1. User asks a question
2. System creates multiple versions of the question
3. Searches documents using each version
4. Combines all results
5. Uses them to generate the answer

It's like asking **3 friends to search Google with different keywords** — together they find more than any one person alone.

In [ ]:
# Fusion RAG (uses same tools — no new libraries!)

def fusion_rag(query):

    # Step 1: Generate 3 different versions of the query
    variations = llm.invoke(f"""
    Generate 3 different versions of this question. Each should ask the same thing but with different wording.
    Return only the 3 questions, one per line. No numbering.

    Original question: {query}
    """).content.strip().split("\n")

    all_queries = [query] + variations
    print("Queries used:")
    for q in all_queries:
        print(f"  - {q}")

    # Step 2: Search with each query and collect all results
    all_docs = []
    seen = set()
    for q in all_queries:
        docs = retriever.invoke(q)
        for doc in docs:
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                all_docs.append(doc)

    print(f"\nFound {len(all_docs)} unique documents (from {len(all_queries)} queries)")

    # Step 3: Combine all unique documents
    context = "\n".join([doc.page_content for doc in all_docs])

    # Step 4: Generate answer
    answer = llm.invoke(f"""
    Answer the question using only this context:
    Context: {context}
    Question: {query}
    """)
    return answer.content

In [ ]:
# Test Fusion RAG
print(fusion_rag("What is RAG and how does it work?"))

## Advanced RAG

Advanced RAG improves basic RAG by making the search smarter and selecting only the most relevant documents before answering.

# How it Works (Simple Steps)
1. User asks a question
2. System rewrites the question to improve search
3. Searches documents using the improved query
4. Re-ranks the documents to keep only the best ones
5. Uses the top documents to generate the answer

In [ ]:
# Advanced RAG — Query Rewriting + Re-Ranking (no new libraries!)

def advanced_rag(query):

    # Step 1: Rewrite the query for better retrieval
    better_query = llm.invoke(f"""
    Rewrite this question to be more specific and detailed for searching documents.
    Return only the rewritten question, nothing else.
    Original: {query}
    """).content.strip()

    print(f"Original query: {query}")
    print(f"Rewritten query: {better_query}\n")

    # Step 2: Retrieve documents using the improved query
    docs = retriever.invoke(better_query)
    print(f"Retrieved {len(docs)} documents")

    # Step 3: Re-rank — LLM scores each document (1-10)
    ranked_docs = []
    for i, doc in enumerate(docs):
        score = llm.invoke(f"""
        Rate how relevant this document is to the question on a scale of 1-10.
        Return only the number.
        Question: {query}
        Document: {doc.page_content}
        """).content.strip()

        try:
            score = int(score)
        except:
            score = 5

        print(f"  Doc {i+1}: score {score}/10 — {doc.page_content[:60]}...")
        ranked_docs.append((score, doc))

    # Sort by score (highest first) and keep top 2
    ranked_docs.sort(key=lambda x: x[0], reverse=True)
    top_docs = [doc for score, doc in ranked_docs[:2]]
    print(f"\nUsing top {len(top_docs)} documents for answer\n")

    # Step 4: Generate answer using only top-ranked documents
    context = "\n".join([doc.page_content for doc in top_docs])
    answer = llm.invoke(f"""
    Answer the question using only this context:
    Context: {context}
    Question: {query}
    """)
    return answer.content

In [ ]:
# Test Advanced RAG
print(advanced_rag("What is RAG and how does it work?"))

## What is Speculative RAG(Draft-First RAG)?

"Speculative RAG" is **not a standard term** — a better name is **Draft-First RAG**.

The idea: Start with a quick answer (draft), then use retrieval to verify and improve it.

# How it Works (Simple Steps)
1. User asks a question
2. System generates a quick initial answer (draft/guess)
3. Uses the original query + draft to retrieve relevant documents
4. Always grounds the final answer in the retrieved documents
5. Returns the improved, document-backed final answer

In [ ]:
# Draft-First RAG — Guess first, then verify with documents

def draft_first_rag(query):

    # Step 1: Generate a draft answer — no documents used
    guess = llm.invoke(f"""
    Answer this question from your own knowledge. Keep it short.
    Question: {query}
    """).content.strip()

    print(f"Draft guess:\n{guess}\n")

    # Step 2: Use BOTH the original query and the guess to retrieve documents
    # This combines the user's original intent with the model's hypothesis
    docs = retriever.invoke(query + " " + guess)
    context = "\n".join([doc.page_content for doc in docs])
    print(f"Retrieved {len(docs)} documents to validate the guess\n")

    # Step 3: Always ground the final answer in the retrieved documents
    final = llm.invoke(f"""
    You made this initial draft:
    Draft: {guess}

    Here are the actual documents:
    Documents: {context}

    Using ONLY the documents above, produce a correct and grounded answer.
    Do not keep any claims from the draft that are not supported by the documents.
    Question: {query}
    """).content.strip()

    return final

In [ ]:
# Test Draft-First RAG
print(draft_first_rag("What is Self-RAG?"))

| Type                              | Core Idea                           | What it Improves        | Simple Flow                                         |
| --------------------------------- | ----------------------------------- | ----------------------- | --------------------------------------------------- |
| **Basic RAG**                     | Retrieve documents and answer       | Adds external knowledge | Query → Retrieve → Answer                           |
| **Self-RAG**                      | Check and improve the answer        | Answer correctness      | Query → Retrieve → Answer → Check → Improve         |
| **CRAG** (Corrective RAG)         | Check and fix retrieved docs        | Document quality        | Query → Retrieve → Check → Fix → Answer             |
| **Fusion RAG**                    | Use multiple query variations       | Retrieval coverage      | Query → Multi-queries → Retrieve → Combine → Answer |
| **Advanced RAG**                  | Improve search and select best docs | Retrieval relevance     | Query → Rewrite → Retrieve → Re-rank → Answer       |
| **Speculative (Draft-first) RAG** | Guess first, then verify            | Speed + direction       | Query → Guess → Retrieve → Refine → Answer          |
